# NQ Intraday Research Notebook

In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1) Setup projet

In [12]:
from pathlib import Path
import sys
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))
print("Project root set to:", root)


Project root set to: D:\Business\Trading\VSCODE\algo-trading-intraday-research


In [13]:
import pandas as pd
import matplotlib.pyplot as plt

from src.config.paths import PROCESSED_DATA_DIR, ensure_directories
from src.data.loader import load_ohlcv_csv
from src.data.validation import validate_ohlcv
from src.data.cleaning import clean_ohlcv
from src.data.session import extract_rth, add_session_date
from src.features.intraday import add_intraday_features, add_session_vwap, add_continuous_session_vwap
from src.features.opening_range import compute_opening_range
from src.features.returns import add_simple_returns, add_log_returns
from src.features.volatility import add_rolling_std, add_atr
from src.strategy.orb import ORBStrategy
from src.engine.execution_model import ExecutionModel
from src.engine.backtester import run_backtest
from src.engine.portfolio import build_equity_curve
from src.analytics.metrics import compute_metrics
from src.analytics.diagnostics import performance_by_weekday, performance_by_month, performance_by_year
from src.analytics.heatmaps import run_orb_grid_search, pivot_heatmap
from src.visualization.plots import plot_trade_histogram, plot_pnl_distribution, plot_cumulative_pnl
from src.visualization.equity import plot_equity_curve, plot_drawdown_curve

## 2) Chargement d’un sample 1-minute

In [ ]:
# api_key = ''

# import databento as db

# # Connexion au service historical
# client = db.Historical(api_key)

# # Récupération du MNQ front month en bougies 1 minute
# data = client.timeseries.get_range(
#     dataset="GLBX.MDP3",      # CME Globex MDP 3.0
#     symbols="NQ.v.0",        # MNQ continuous front month
#     stype_in="continuous",
#     schema="ohlcv-1m",
#     start="2019-02-05T00:00:00Z",
#     end="2026-03-20T0:00:00Z",
# )

# # Conversion en DataFrame pandas
# df = data.to_df()

In [15]:
# df.to_parquet(DOWNLOADED_DATA_DIR / 'NQ_1mim_part1.parquet')

In [16]:
df = pd.read_parquet(PROCESSED_DATA_DIR / 'parquet' / 'MNQ_c_0_1m_20260321_094501.parquet')

In [17]:
df.head()

,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol
timestamp,,,,,,,,,
2019-05-05 18:03:00-04:00,33,1,8078,7748.75,7748.75,7748.75,7748.75,1,MNQ.c.0
2019-05-05 18:12:00-04:00,33,1,8078,7713.75,7713.75,7713.75,7713.75,1,MNQ.c.0
2019-05-05 18:17:00-04:00,33,1,8078,7713.75,7713.75,7713.75,7713.75,2,MNQ.c.0
2019-05-05 18:23:00-04:00,33,1,8078,7713.50,7713.50,7713.50,7713.50,1,MNQ.c.0
2019-05-05 18:25:00-04:00,33,1,8078,7718.25,7718.25,7718.25,7718.25,1,MNQ.c.0


## 3) Contrôle qualité de la donnée

In [18]:
quality_report = validate_ohlcv(df)
quality_report

QualityReport(rows=2401697, missing_required_columns=['timestamp'], is_chronological=False, duplicate_timestamps=0, invalid_ohlc_rows=0, negative_volume_rows=0)

## 4) Nettoyage

In [19]:
df = clean_ohlcv(df)
len(df), df.head()

KeyError: Index(['timestamp'], dtype='object')

## 5) Filtrage de session (RTH)

In [ ]:
df_rth = extract_rth(df)
df_rth = add_session_date(df_rth)
df_rth[['timestamp', 'session_date']].head()

## 6) Construction de features intraday

In [ ]:
df_feat = add_intraday_features(df)
df_feat = add_simple_returns(df_feat)
df_feat = add_log_returns(df_feat)
df_feat = add_rolling_std(df_feat, window=5)
df_feat = add_atr(df_feat, window=5)
# EMA is now computed dynamically by ORBStrategy if needed
# df_feat = add_ema(df_feat, window=30)
df_feat = add_session_vwap(df_feat)
df_feat = add_continuous_session_vwap(df_feat, session_start_hour=18)
df_feat[["timestamp", "session_date", "continuous_session_date", "session_vwap", "continuous_session_vwap"]].tail(20)

## 7) Calcul de l’opening range

In [ ]:
or_minutes=  15
opening_time = '09:30:00'
df_or = compute_opening_range(df_feat, or_minutes=or_minutes, opening_time=opening_time)
df_or[['timestamp', 'session_date', 'or_high', 'or_low', 'or_width', 'or_midpoint']].head(10)

## 8) Backtest ORB baseline

In [ ]:
from src.config.settings import DEFAULT_INITIAL_CAPITAL_USD


strategy = ORBStrategy(
    or_minutes=or_minutes,
    direction='long',
    one_trade_per_day=True,
    entry_buffer_ticks=2,
    stop_buffer_ticks=2,
    target_multiple=2.0,
    opening_time=opening_time,
    vwap_confirmation=True,
    vwap_column='continuous_session_vwap',
    time_exit='16:00:00',
    account_size_usd=DEFAULT_INITIAL_CAPITAL_USD,  # for paper-style fixed contract sizing, remove risk sizing
    risk_per_trade_pct=0.5,
)
df_sig = strategy.generate_signals(df_or)

In [ ]:
df_sig[df_sig['signal'] != 0]

In [ ]:
trades = run_backtest(
    df_sig,
    execution_model=ExecutionModel(),
    time_exit=strategy.time_exit,
    stop_buffer_ticks=strategy.stop_buffer_ticks,
    target_multiple=strategy.target_multiple,
    account_size_usd=strategy.account_size_usd,
    risk_per_trade_pct=strategy.risk_per_trade_pct,
    entry_on_next_open=True,  # use realistic next-close entry after signal
)


In [ ]:
trades[[
    'session_date',
    'direction',
    'quantity',
    'risk_budget_usd',
    'actual_risk_usd',
    'net_pnl_usd',
]].head()

In [ ]:
print(trades['exit_reason'].value_counts())

## 9) Analyse des trades

In [ ]:
metrics = compute_metrics(
    trades,
    signal_df=df_sig,
    session_dates=df_feat["session_date"].unique(),
)
metrics

In [ ]:
performance_by_weekday(trades)

In [ ]:
performance_by_month(trades)

In [ ]:
performance_by_year(trades)

## 10) Equity curve

In [ ]:
equity = build_equity_curve(trades, initial_capital=strategy.account_size_usd or 100_000.0)
equity.head()

In [ ]:
if not trades.empty:
    plot_trade_histogram(trades)
    plot_pnl_distribution(trades)
    plot_cumulative_pnl(trades)
    plot_equity_curve(equity)
    plot_drawdown_curve(equity)
    plt.show()

## 11) Tracer une journée spécifique pour confirmation visuelle

In [ ]:
# Tracé rapide de bougies pour une date donnée
plot_date = "2026-03-02"      # format YYYY-MM-DD
display_tz = "America/New_York"
session_only = True
session_start = "09:30"
session_end = "16:00"
start_time = None             # ex: "10:00" pour démarrer plus tard
n_bars = None                   # None pour afficher toute la session

source_df = df.copy() if "df" in globals() else df_raw.copy()

ts = pd.to_datetime(source_df["timestamp"])
if ts.dt.tz is None:
    ts_local = ts.dt.tz_localize(display_tz)
else:
    ts_local = ts.dt.tz_convert(display_tz)

plot_df = source_df.assign(timestamp_local=ts_local)
target_date = pd.Timestamp(plot_date).date()
plot_df = plot_df.loc[plot_df["timestamp_local"].dt.date == target_date].copy()

if session_only:
    hhmm = plot_df["timestamp_local"].dt.strftime("%H:%M")
    plot_df = plot_df.loc[(hhmm >= session_start) & (hhmm <= session_end)].copy()

if start_time:
    start_dt = pd.Timestamp(f"{plot_date} {start_time}").tz_localize(display_tz)
    plot_df = plot_df.loc[plot_df["timestamp_local"] >= start_dt].copy()

if n_bars is not None:
    plot_df = plot_df.head(n_bars).copy()

if plot_df.empty:
    raise ValueError(f"Aucune bougie trouvée pour {plot_date} avec les filtres actuels.")

colors = ["#16a34a" if close_ >= open_ else "#dc2626" for open_, close_ in zip(plot_df["open"], plot_df["close"])]
body = plot_df["close"] - plot_df["open"]
price_range = float(plot_df["high"].max() - plot_df["low"].min())
min_body = max(price_range * 0.001, 0.01)
body = body.mask(body.eq(0), min_body)

symbol = plot_df["symbol"].dropna().iloc[0] if "symbol" in plot_df.columns else "Instrument"

fig, ax = plt.subplots(figsize=(max(12, len(plot_df) * 0.18), 15))
x = list(range(len(plot_df)))

ax.vlines(x, plot_df["low"], plot_df["high"], color=colors, linewidth=1.0, alpha=0.9)
ax.bar(x, body, bottom=plot_df["open"], color=colors, width=0.7, align="center")

labels = plot_df["timestamp_local"].dt.strftime("%H:%M")
tick_step = max(1, len(plot_df) // 12)
ax.set_xticks(x[::tick_step])
ax.set_xticklabels(labels.iloc[::tick_step], rotation=45, ha="right")
ax.set_title(f"{symbol} | {plot_date} | {display_tz}")
ax.set_ylabel("Prix")
ax.grid(axis="y", linestyle=":", alpha=0.35)

plt.tight_layout()
plt.show()

plot_df[["timestamp_local", "open", "high", "low", "close", "volume"]].head(10)
